# 06 — Reasoning-Oriented Prompting

A technical incident needs an evidence-backed recommendation. This offline lab compares direct answering with an observable plan and verifier.

## Boundary

Plans, assumptions, and verification results are inspectable artifacts. Do not request private chain-of-thought. No APIs or production actions are used.

In [1]:
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path
import sys
path=Path.cwd()/'curriculum/intermediate/06-reasoning-oriented-prompting/lab.py'
if not path.exists(): path=Path.cwd()/'lab.py'
spec=spec_from_file_location('reasoning_lab',path); lab=module_from_spec(spec); sys.modules[spec.name]=lab; spec.loader.exec_module(lab)

## Baseline

A direct response suggests restarting the service. It has no evidence contract and is unsafe for the incident.

In [2]:
baseline=lab.direct_answer(lab.INCIDENT)
lab.score(baseline), baseline

({'supported': False, 'calls': 1, 'safe': False},
 {'recommendation': 'restart the service', 'supported': False, 'calls': 1})

## Planner and verifier experiment

The improved path records the bounded plan, one assumption, evidence check, and recommendation. Compare the same incident rather than changing several variables.

In [3]:
improved=lab.plan_and_verify(lab.INCIDENT)
assert improved['verification']
lab.score(improved), improved

({'supported': True, 'calls': 2, 'safe': True},
 {'plan': ('classify symptom',
   'check evidence',
   'recommend least-risk action'),
  'assumption': 'database outage is possible',
  'recommendation': 'escalate database incident',
  'verification': True,
  'supported': True,
  'calls': 2})

## Failure and production upgrade

Remove database evidence: the correct outcome is collecting a trace, not a confident diagnosis. Bound calls and retries; evaluate support, task success, cost, and latency on held-out incidents.

Exercises: add conflicting evidence, a verifier failure, and a deterministic health-check alternative.

In [4]:
missing=lab.Incident(('checkout failures',),())
assert lab.plan_and_verify(missing)['recommendation']=='collect database trace'